# Assignment 1: Vector Database Creation and Retrieval
## Day 6 Session 2 - RAG Fundamentals

The goal is not to build a completely new system from scratch. The goal is to repeat the same flow with guided TODOs so the first notebook feels intuitive.

## What you will build

You will build a small multimodal RAG retrieval pipeline using LlamaIndex and LanceDB:

1. Mount Google Drive and install dependencies
2. Configure LlamaIndex settings
3. Explore a folder containing different file types
4. Load documents using `SimpleDirectoryReader`
5. Create a LanceDB vector store
6. Create a `StorageContext`
7. Build a `VectorStoreIndex`
8. Retrieve relevant chunks for a query
9. Inspect retrieved chunks and metadata
10. Optionally generate an answer using a query engine

## Main idea

Yesterday's session and notebooks showed the full RAG system end-to-end. This assignment asks you to complete the similar flow step by step.

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it

## 0. Mount Google Drive

Use this if you are running the notebook in Google Colab.

In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

## 1. Install dependencies

Use the same requirements file path used in class. If your folder path is different, update it below.

In [ ]:
!pip install -r "C:\Users\devan\ai-accelerator-C7\Day_6_Session_2\session_2\requirements.txt"
#pip install -r "C:\Users\devan\ai-accelerator-C7\Day_6_Session_2\session_2\assignments\requirements.txt"
#!pip install -q uv
#!uv pip install --system -r /content/drive/MyDrive/outskill_c4/requirements.txt

  Using cached google_api_core-2.31.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached google_api_python_client-2.197.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached google_auth-2.53.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached google_auth_httplib2-0.4.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached websockets-15.0.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached lancedb-0.33.0-cp39-abi3-win_amd64.whl.metadata (5.0 kB)
  Using cached llama_index_vector_stores_lancedb-0.5.0-py3-none-any.whl.metadata (461 bytes)
  Using cached llama_index_embeddings_huggingface-0.7.0-py3-none-any.whl.metadata (459 bytes)
  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached openai_whisper-20250625.tar.gz (803 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): 


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Set OpenRouter API key

We use OpenRouter for the LLM, similar to the class notebook. The embedding model will be local.

In [10]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

if api_key:
    print("✓ OpenRouter key loaded successfully")
else:
    print("✗ OPENROUTER_API_KEY not found")

#import os
#from getpass import getpass

#os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter key: ")
#print("OpenRouter key set successfully")

✓ OpenRouter key loaded successfully


## 3. Imports and configuration

Before running the full pipeline, check these values:

- `data_path`: folder that contains your input files (CHANGE IT ACCORDING TO YOUR OWN PATH)
- `vector_db_path`: where LanceDB will store vectors
- `index_storage_path`: where LlamaIndex will persist index information
- `chunk_size` and `chunk_overlap`: same concepts discussed in class

In [11]:
from pathlib import Path
print(Path.cwd())

c:\Users\devan\ai-accelerator-C7\Day_6_Session_2\session_2\assignments


In [12]:
from pathlib import Path
import os

BASE_DIR = Path.cwd()

CONFIG = {
    "llm_model": "gpt-5-mini",
    "embedding_model": "local:BAAI/bge-small-en-v1.5",
    "chunk_size": 512,
    "chunk_overlap": 50,
    "similarity_top_k": 5,

    "data_path": str(BASE_DIR.parent.parent / "data"),
    "vector_db_path": str(BASE_DIR.parent.parent / "storage" / "assignment_multimodal_vectordb"),
    "index_storage_path": str(BASE_DIR.parent.parent / "storage" / "assignment_multimodal_index"),

    "table_name": "assignment_documents"
}

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Configuration loaded")
print("Data path:", CONFIG["data_path"])
print("Vector DB path:", CONFIG["vector_db_path"])
print("Index storage path:", CONFIG["index_storage_path"])

Configuration loaded
Data path: c:\Users\devan\ai-accelerator-C7\Day_6_Session_2\data
Vector DB path: c:\Users\devan\ai-accelerator-C7\Day_6_Session_2\storage\assignment_multimodal_vectordb
Index storage path: c:\Users\devan\ai-accelerator-C7\Day_6_Session_2\storage\assignment_multimodal_index


## 4. Configure LlamaIndex settings

This step defines:

- which LLM will generate answers
- which embedding model will convert text into vectors
- how text will be chunked

### Your task

Complete the function below.

Hint: This is very similar to the `configure_llamaindex_settings()` function from the class notebook.

In [18]:
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.embeddings import resolve_embed_model
from llama_index.llms.openrouter import OpenRouter

print("All imports successful")

All imports successful


In [19]:
configure_llamaindex_settings()

✓ LLM configured: gpt-5-mini


c:\Users\devan\ai-accelerator-C7\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Embedding model configured: local:BAAI/bge-small-en-v1.5
✓ Text chunking configured: 512 chars with 50 overlap


In [20]:
def configure_llamaindex_settings():
    """Configure LlamaIndex global settings using hardcoded configuration."""

    # Set up LLM with OpenRouter using hardcoded model
    Settings.llm = OpenRouter(
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model=CONFIG["llm_model"]
    )
    print(f"✓ LLM configured: {CONFIG['llm_model']}")

    # Set up local embedding model (downloads locally first time, then cached)
    Settings.embed_model = resolve_embed_model(CONFIG["embedding_model"])
    print(f"✓ Embedding model configured: {CONFIG['embedding_model']}")

    # Set up node parser for chunking with hardcoded settings
    Settings.node_parser = SentenceSplitter(
        chunk_size=CONFIG["chunk_size"],
        chunk_overlap=CONFIG["chunk_overlap"]
    )
    print(f"✓ Text chunking configured: {CONFIG['chunk_size']} chars with {CONFIG['chunk_overlap']} overlap")

# Configure the settings
configure_llamaindex_settings()
print("✓ LlamaIndex settings configured for multimodal processing")

✓ LLM configured: gpt-5-mini
✓ Embedding model configured: local:BAAI/bge-small-en-v1.5
✓ Text chunking configured: 512 chars with 50 overlap
✓ LlamaIndex settings configured for multimodal processing


## 5. Explore the dataset

Before loading documents, always inspect what files are present.

This helps you answer questions like:

- How many files are present?
- What file types are present?
- Are we working with PDFs only or multiple formats?
- Is the folder path correct?

In [36]:
from pathlib import Path

def explore_dataset(data_path: str = None):
    """
    Explore and categorize the files in our dataset by type.
    """

    if data_path is None:
        data_path = CONFIG["data_path"]

    data_dir = Path(data_path)

    if not data_dir.exists():
        print(f"Data directory not found: {data_dir}")
        return {}, []

    file_types = {}
    all_files = []

    for file_path in data_dir.rglob("*"):
        if file_path.is_file():

            suffix = file_path.suffix.lower()
            file_size = file_path.stat().st_size

            if suffix not in file_types:
                file_types[suffix] = []

            file_info = {
                "path": str(file_path),
                "name": file_path.name,
                "size_mb": round(file_size / (1024 * 1024), 2),
                "size_bytes": file_size
            }

            file_types[suffix].append(file_info)
            all_files.append(file_info)

    print("--- Dataset Overview ---")
    print(f"Total files found: {len(all_files)}")

    print("\nFile Types Distribution:")

    for file_type, files in sorted(file_types.items()):
        if file_type:
            total_size = sum(f["size_mb"] for f in files)
            print(f"{file_type}: {len(files)} files ({total_size:.2f} MB)")

    return file_types, all_files

In [37]:
print("explore_dataset" in globals())

True


In [38]:
file_types, all_files = explore_dataset()

print(f"\n✓ Found {len(all_files)} files across {len(file_types)} different file types")

--- Dataset Overview ---
Total files found: 21

File Types Distribution:
.csv: 4 files (0.00 MB)
.html: 2 files (0.00 MB)
.md: 4 files (0.00 MB)
.mp3: 3 files (2.95 MB)
.pdf: 2 files (1.92 MB)
.png: 6 files (0.55 MB)

✓ Found 21 files across 6 different file types


## 6. Load multimodal documents

In the class notebook, we used `SimpleDirectoryReader` because it can load many file types such as PDF, CSV, Markdown, HTML, images, and notebooks.

At this stage, files become LlamaIndex `Document` objects.

### Your task

Complete the function below to load documents from the dataset folder.

In [39]:
from llama_index.core import SimpleDirectoryReader

def load_documents_from_folder(data_path: str):
    """
    Load documents from a folder using SimpleDirectoryReader.

    TODO: Complete this function to load documents from the given folder path.
    HINT: Use SimpleDirectoryReader with recursive parameter to load all files

    Args:
        data_path (str): Path to the folder containing data

    Returns:
        List of documents loaded from the folder
    """
    # TODO: Create SimpleDirectoryReader (HINT: use recursive=True)
    # reader = ?
    reader = SimpleDirectoryReader(
        input_dir=data_path,
        recursive=True
    )

    # TODO: Load and return documents
    # documents = ?
    documents = reader.load_data()

    # return documents
    return documents

# Test the function after you complete it
test_folder = CONFIG["data_path"]
documents = load_documents_from_folder(test_folder)
print(f"Loaded {len(documents)} documents")

Loaded 21 documents


## 7. Create LanceDB vector store

The vector store is where embeddings are stored and searched.

Important idea:

- LlamaIndex creates chunks and embeddings
- LanceDB stores the vector representation
- Later, the retriever searches this vector store

### Your task

Complete the function below.

In [40]:
import lancedb
# Vector store and index creation
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

def create_vector_store(vector_db_path: str, table_name: str):
    """
    Create a LanceDB vector store for storing document embeddings.

    TODO: In this function, you need to:
    1. Create the database directory if it does not already exist.
    2. Create a LanceDBVectorStore object.
    3. Return the vector store.

    Args:
        db_path (str): Path where the vector database will be stored.
        table_name (str): Name of the table in the vector database.

    Returns:
        LanceDBVectorStore: Configured vector store.
    """

    # TODO: Create the directory if it doesn't exist
    # Path(db_path).mkdir(parents=True, exist_ok=True)
    Path(vector_db_path).mkdir(parents=True, exist_ok=True)

    # TODO: Connect to LanceDB (creates a connection to the LanceDB database)
    db = lancedb.connect(str(vector_db_path))

    # TODO: Create vector store (HINT: Use LanceDBVectorStore)
    # vector_store = ?
    vector_store = LanceDBVectorStore(
        uri=str(vector_db_path),
        table_name=table_name
    )

    # print("✓ LanceDB vector store created for multimodal data")
    print("✓ LanceDB vector store created for multimodal data")

    # return vector_store
    return vector_store


# Test the function after you complete it
vector_store = create_vector_store(
    vector_db_path=CONFIG["vector_db_path"],
    table_name=CONFIG["table_name"]
)

print(f"Vector store created: {vector_store is not None}")

✓ LanceDB vector store created for multimodal data
Vector store created: True


## 8. Create StorageContext and VectorStoreIndex

This is one of the most important parts of the assignment.

### What is `StorageContext`?

Think of it as the object that tells LlamaIndex where the prepared index data should live.

In this assignment:

- `vector_store` stores embeddings
- `StorageContext` connects LlamaIndex to that vector store
- `VectorStoreIndex.from_documents()` builds the index from documents

### Your task

Complete the function below.

In [41]:
def create_vector_index(documents: List, vector_store, persist_dir: str = None):
    """
    Create a VectorStoreIndex from loaded documents.

    Args:
        documents: Loaded LlamaIndex documents.
        vector_store: LanceDB vector store.
        persist_dir: Folder for persisting index metadata.

    Returns:
        VectorStoreIndex object.
    """
    if persist_dir is None:
        persist_dir = CONFIG["index_storage_path"]

    if not documents:
        raise ValueError("No documents found. Load documents before creating the index.")

    Path(persist_dir).mkdir(parents=True, exist_ok=True)

    print("Creating storage context")

    # TODO: Create storage context from vector_store (HINT: Use StorageContext)
    # storage_context = ?
    storage_context = StorageContext.from_defaults(
        vector_store=vector_store
    )

    # TODO: Create the VectorStoreIndex from documents
    # index = ?
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )

    # TODO: Persist index metadata
    # index.storage_context.persist(persist_dir=persist_dir)
    index.storage_context.persist(
        persist_dir=persist_dir
    )

    # return index
    return index


if vector_store and documents:
    index = create_vector_index(
        documents,
        vector_store
    )

Creating storage context


Generating embeddings: 100%|██████████| 1607/1607 [02:00<00:00, 13.33it/s]


## 9. Create a retriever
A retriever does not generate an answer. It only returns the most relevant chunks.

This is useful because it lets you inspect what the RAG system found before the LLM answers.

### Your task

Complete the retriever setup.

In [42]:
from llama_index.core.retrievers import VectorIndexRetriever

def create_retriever(index, similarity_top_k: int = None):
    """
    Create a retriever from the index.

    Args:
        index: VectorStoreIndex object.
        similarity_top_k: Number of chunks to retrieve.

    Returns:
        VectorIndexRetriever object.
    """
    if similarity_top_k is None:
        similarity_top_k = CONFIG["similarity_top_k"]

    # TODO: Create VectorIndexRetriever
    # retriever = ?
    retriever = VectorIndexRetriever(
        index=index,
        similarity_top_k=similarity_top_k
    )

    # print(f"Retriever created with similarity_top_k={similarity_top_k}")
    print(f"Retriever created with similarity_top_k={similarity_top_k}")

    # return retriever
    return retriever

retriever = create_retriever(index)

Retriever created with similarity_top_k=5


## 10. Retrieve chunks for a query

This step helps you see exactly what semantic search returns.

Good queries to try:

- Ask about a topic you know exists in the dataset
- Ask using different wording than the original document
- Ask a broad question and inspect whether results are noisy

In [46]:
def retrieve_chunks(retriever, query: str, show_text: bool = True):
    """
    Retrieve relevant chunks for a query and print their metadata and score.

    Args:
        retriever: VectorIndexRetriever object.
        query: User query.
        show_text: Whether to print text previews.

    Returns:
        Retrieved nodes.
    """
    print("Query:", query)
    print("=" * 80)

    nodes = retriever.retrieve(query)

    print("Retrieved chunks:", len(nodes))

    for i, node_with_score in enumerate(nodes, 1):
        node = node_with_score.node
        score = node_with_score.score
        metadata = node.metadata

        print(f"Result {i}")
        print("Score:", score)
        print("File name:", metadata.get("file_name", "unknown"))
        print("File type:", metadata.get("file_type", "unknown"))

        if show_text:
            print("Text preview:")
            print(node.get_content()[:700])

        print("-" * 80)

    return nodes

sample_query = "Where should I travel next in May-June?"
retrieved_nodes = retrieve_chunks(retriever, sample_query)

Query: Where should I travel next in May-June?
Retrieved chunks: 5
Result 1
Score: 0.5448606014251709
File name: city_guides.md
File type: text/markdown
Text preview:
# Ultimate City Travel Guide

## Paris, France 🇫🇷

**Best Time to Visit:** April-June, September-October
**Must-See Attractions:**
- Eiffel Tower - Iconic iron lattice tower
- Louvre Museum - World's largest art museum
- Notre-Dame Cathedral - Gothic masterpiece
- Champs-Élysées - Famous shopping avenue

**Local Cuisine:** Croissants, Escargot, Coq au Vin, Macarons
**Transportation:** Metro system, Vélib bike sharing
**Budget:** €100-150 per day for mid-range travel

---

## Tokyo, Japan 🇯🇵

**Best Time to Visit:** March-May (cherry blossoms), September-November
**Must-See Attractions:**
- Senso-ji Temple - Ancient Buddhist temple
- Shibuya Crossing - World's busiest pe
--------------------------------------------------------------------------------
Result 2
Score: 0.5447008609771729
File name: city_guides.md
File type: t

## 11. Build a query engine

A retriever only returns chunks. A query engine uses the retriever and an LLM to generate a final response.

This mirrors the class notebook section where we created a multimodal query engine.

### Your task

Complete the function below.

In [63]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

def create_query_engine(retriever, similarity_top_k: int = None):
    """
    Create a RetrieverQueryEngine using a retriever.

    Args:
        retriever: VectorIndexRetriever object.
        similarity_top_k: Number of chunks to retrieve.

    Returns:
        RetrieverQueryEngine object.
    """
    if similarity_top_k is None:
        similarity_top_k = CONFIG["similarity_top_k"]

    response_synthesizer = get_response_synthesizer(
        response_mode="compact"
    )

    # TODO: Create query engine (HINT: Use RetrieverQueryEngine)
    # query_engine = ?
    query_engine = RetrieverQueryEngine(
        retriever=retriever,
        response_synthesizer=response_synthesizer
    )

    print("Query engine created")

    return query_engine

query_engine = create_query_engine(retriever)

Query engine created


In [62]:
print(Settings.llm)

response = Settings.llm.complete("Say hello")

print(response)

callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x0000018A9C09EFD0> rate_limiter=None system_prompt=None messages_to_prompt=<function messages_to_prompt at 0x0000018AF5622140> completion_to_prompt=<function default_completion_to_prompt at 0x0000018AF5AEF690> output_parser=None pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'> query_wrapper_prompt=None model='gpt-5-mini' temperature=1.0 max_tokens=256 logprobs=None top_logprobs=0 additional_kwargs={} max_retries=5 timeout=60.0 default_headers=None reuse_client=True api_key='REDACTED' api_base='https://openrouter.ai/api/v1' api_version='' strict=False reasoning_effort=None modalities=None audio_config=None context_window=3900 is_chat_model=True is_function_calling_model=False should_use_structured_outputs=False tokenizer=None
Hello! How can I help you today?


## 12. Ask a question and inspect sources

This is the final RAG step.

We ask a question, get a generated answer, and then inspect which source chunks were used.

In [64]:
def ask_question(query_engine, question: str, show_sources: bool = True):
    """
    Ask a question to the query engine and display answer plus sources.

    Args:
        query_engine: RetrieverQueryEngine object.
        question: User question.
        show_sources: Whether to show source chunks.
    """
    print("Question:", question)
    print("=" * 80)

    response = query_engine.query(question)

    print("Answer:")
    print(str(response))

    if show_sources:
        print("Sources used:")
        source_nodes = getattr(response, "source_nodes", [])
        for i, source in enumerate(source_nodes, 1):
            node = source.node
            metadata = node.metadata
            print(f"Source {i}")
            print("Score:", source.score)
            print("File name:", metadata.get("file_name", "unknown"))
            print("File type:", metadata.get("file_type", "unknown"))
            print("Text preview:")
            print(node.get_content()[:500])
            print("-" * 80)

# Try your own question here
question = "What are the steps to make Carbonara?"
ask_question(query_engine, question, show_sources=True)

Question: What are the steps to make Carbonara?
Answer:
Empty Response
Sources used:
Source 1
Score: 0.5940585136413574
File name: recipe_instructions.md
File type: text/markdown
Text preview:
# 🍝 Classic Spaghetti Carbonara Recipe

## Ingredients
- 400g spaghetti pasta
- 4 large egg yolks
- 100g pecorino romano cheese (grated)
- 150g guanciale or pancetta (diced)
- Black pepper (freshly ground)
- Salt for pasta water

## Instructions

### Step 1: Prepare the Sauce
1. In a large bowl, whisk together egg yolks and grated pecorino cheese
2. Add plenty of freshly ground black pepper
3. Mix until smooth and creamy (no lumps)

### Step 2: Cook the Guanciale
1. Heat a larg
--------------------------------------------------------------------------------
Source 2
Score: 0.5892278552055359
File name: recipe_instructions.md
File type: text/markdown
Text preview:
# 🍝 Classic Spaghetti Carbonara Recipe

## Ingredients
- 400g spaghetti pasta
- 4 large egg yolks
- 100g pecorino romano cheese (grated

In [65]:
response = index.as_query_engine(
    similarity_top_k=5
).query(
    "What are the steps to make Carbonara?"
)

print(response)

Empty Response


In [57]:
response = index.as_query_engine(
    similarity_top_k=5
).query(
    "What are the steps to make Carbonara?"
)

print("Response object:", response)
print("Response text:", response.response)
print("Source nodes:", len(response.source_nodes))

Response object: Empty Response
Response text: Empty Response
Source nodes: 5


In [58]:
from llama_index.core import Settings

nodes = retriever.retrieve("What are the steps to make Carbonara?")

for i, n in enumerate(nodes[:2], 1):
    print(f"\nNode {i}")
    print(n.node.get_content()[:1000])


Node 1
# 🍝 Classic Spaghetti Carbonara Recipe

## Ingredients
- 400g spaghetti pasta
- 4 large egg yolks
- 100g pecorino romano cheese (grated)
- 150g guanciale or pancetta (diced)
- Black pepper (freshly ground)
- Salt for pasta water

## Instructions

### Step 1: Prepare the Sauce
1. In a large bowl, whisk together egg yolks and grated pecorino cheese
2. Add plenty of freshly ground black pepper
3. Mix until smooth and creamy (no lumps)

### Step 2: Cook the Guanciale
1. Heat a large pan over medium heat
2. Add diced guanciale (no oil needed)
3. Cook until crispy and golden (about 5-7 minutes)
4. Reserve the rendered fat in the pan

### Step 3: Cook the Pasta
1. Bring a large pot of salted water to boil
2. Add spaghetti and cook until al dente (about 10-12 minutes)
3. Reserve 1 cup of pasta water before draining

### Step 4: Combine Everything
1. Add hot, drained pasta to the pan with guanciale
2. Remove from heat immediately
3. Quickly toss pasta with the ren

Node 2
# 🍝 Classic Sp

In [59]:
from llama_index.core import Settings

response = Settings.llm.complete("""
Using the following context answer the question.

Context:
Classic Carbonara Recipe:
1. Prepare egg and cheese mixture.
2. Cook guanciale.
3. Cook spaghetti.
4. Combine everything.
5. Serve immediately.

Question:
What are the steps to make Carbonara?
""")

print(response)

1. Prepare egg and cheese mixture.  
2. Cook guanciale.  
3. Cook spaghetti.  
4. Combine everything.  
5. Serve immediately.


## Conclusion

🎉 **Congratulations!** You have successfully built an advanced **Multimodal RAG System** using LlamaIndex's `SimpleDirectoryReader` with comprehensive cross-modal capabilities.

## Student experiments (for trying out later after the session)

Complete these small experiments later to build intuition.

### Experiment 1: Change `similarity_top_k`

Try values like 2, 5, and 10.

Question to answer:

- Does increasing `top_k` always improve the answer?
- Do you see more noise when `top_k` is high?

### Experiment 2: Ask the same question in different wording

Example:

- "What is the refund policy?"
- "Can customers get their money back?"

Question to answer:

- Does semantic search still retrieve similar chunks?

### Experiment 3: Inspect sources before trusting the answer

Question to answer:

- Did the LLM answer using the right sources?
- Were any irrelevant chunks included?

In [ ]:
# Experiment area
# Change the query and top_k values below.

experiment_query = "Replace this with your own question"
experiment_top_k = 3

experiment_retriever = create_retriever(index, similarity_top_k=experiment_top_k)
experiment_nodes = retrieve_chunks(experiment_retriever, experiment_query)

## Reflection questions

Answer these in a markdown cell below after you complete the notebook.

1. Why do we parse and load files before indexing?
2. Why do we chunk text instead of storing the full document as one unit?
3. What does `chunk_overlap` help with?
4. What is stored in the vector database?
5. What role does `StorageContext` play?
6. What is the difference between retriever output and query engine output?
7. Give one example where returning retrieved chunks directly is better than generating an answer.
8. Give one example where generation is useful after retrieval.

In [ ]:
# Write short answers here as comments or create a markdown cell below.

# 1.
# 2.
# 3.
# 4.
# 5.
# 6.
# 7.
# 8.